# Branch A — BiLSTM Training
Run on GPU runtime in Google Colab.

In [ ]:
# Cell 1 — Pull latest code
import sys
mods = [k for k in sys.modules if 'src' in k]
for m in mods: del sys.modules[m]
!cd /content/hallucination-detector && git pull origin main
sys.path.insert(0, '/content/hallucination-detector')
print('Done.')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q sentence-transformers
print('Done.')

In [ ]:
# Cell 3 — Upload processed data
from google.colab import files
import zipfile, os
print('Upload your processed_data.zip file:')
uploaded = files.upload()
with zipfile.ZipFile('processed_data.zip', 'r') as z:
    z.extractall('/content/hallucination-detector/data/processed')
print('Files extracted:')
print(os.listdir('/content/hallucination-detector/data/processed'))

In [ ]:
# Cell 4 — Load data and check device
import numpy as np
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

base = '/content/hallucination-detector/data/processed'
X_train = np.load(f'{base}/X_train.npy')
X_val   = np.load(f'{base}/X_val.npy')
y_train = np.load(f'{base}/y_train.npy')
y_val   = np.load(f'{base}/y_val.npy')

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'Classes: {dict(zip(*np.unique(y_train, return_counts=True)))}')

In [ ]:
# Cell 5 — Run hyperparameter search
from src.lstm_model import run_hyperparameter_search

save_dir = '/content/hallucination-detector/models/lstm'
best_config, results = run_hyperparameter_search(
    X_train, X_val, y_train, y_val,
    device=device,
    save_dir=save_dir
)
print(f'\nBest config: {best_config}')

In [ ]:
# Cell 6 — Train final model with best config for more epochs
from src.lstm_model import (
    BiLSTMClassifier, prepare_dataloaders, train_model
)

final_config = best_config.copy()
final_config['epochs']  = 60
final_config['patience'] = 7

train_loader, val_loader = prepare_dataloaders(
    X_train, X_val, y_train, y_val,
    batch_size=final_config['batch_size']
)

model = BiLSTMClassifier(
    input_dim=384,
    hidden_dim=final_config['hidden_dim'],
    num_layers=2,
    num_classes=4,
    dropout=final_config['dropout']
).to(device)

history = train_model(
    model, train_loader, val_loader,
    final_config, device,
    save_path=f'{save_dir}/lstm_final.pt'
)
print('Final model training complete.')

In [ ]:
# Cell 7 — Plot training history
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.savefig('/content/hallucination-detector/models/lstm/training_plot.png', dpi=150)
plt.show()
print('Plot saved.')

In [ ]:
# Cell 8 — Download trained model
import shutil
from google.colab import files
shutil.make_archive('lstm_model', 'zip', '/content/hallucination-detector/models/lstm')
files.download('lstm_model.zip')
print('Download started.')